# 01 Data Pull And Audit

Load the raw match dataset, validate the schema, define the target, and document the strict pre-match feature boundary.

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks" and not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


def load_matches_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [col.strip() for col in df.columns]
    df["date"] = pd.to_datetime(df["date"], errors="raise")
    df["patch"] = df["patch"].fillna("").astype(str).str.strip()
    df["event"] = df["event"].fillna("unknown").astype(str).str.strip()
    df["blue_team"] = df["blue_team"].astype(str).str.strip()
    df["red_team"] = df["red_team"].astype(str).str.strip()
    df["winner"] = df["winner"].astype(str).str.strip()
    df["blue_team_win"] = (df["winner"] == df["blue_team"]).astype(int)
    return df.sort_values(["date"], kind="mergesort").reset_index(drop=True)[
        [
            "season",
            "date",
            "event",
            "patch",
            "blue_team",
            "red_team",
            "winner",
            "blue_team_win",
        ]
    ]


DATA_PATH = PROJECT_ROOT / "data" / "matchs_stats.csv"
df = load_matches_csv(DATA_PATH)
df.to_csv(ARTIFACTS_DIR / "clean_matches.csv", index=False)
df.head()

In [ ]:
df.shape, df["date"].min(), df["date"].max(), df["blue_team_win"].dtype

In [ ]:
blocked_prefixes = ("ban_", "pick_", "top_", "jungler_", "mid_", "adc_", "support_")
blocked_exact = {"winner", "blue_team_win"}
allowed_columns = [
    col for col in df.columns
    if col not in blocked_exact and not col.startswith(blocked_prefixes)
]
blocked_columns = [col for col in df.columns if col not in allowed_columns]
allowed_columns, blocked_columns[:20]

In [ ]:
df.isna().mean().sort_values(ascending=False).head(20)

In [ ]:
df["blue_team_win"].value_counts(dropna=False).sort_index()